# Notebook 02 — Sincronização Lakehouse → Azure Cosmos DB for NoSQL

Segue o modelo de **Análise de Agente com Power BI e Microsoft Fabric**:
o dado curado do cruzamento ANP (produção × importações/exportações × fases)
é materializado como documentos JSON no Cosmos DB, servindo de memória
estruturada para agentes de IA (Fabric Data Agent / Azure AI Foundry) e para
consultas de baixa latência.

**Autenticação**: Entra ID (Service Principal ou identidade gerenciada).
Nenhuma chave primária é usada no código.

**Pré-requisitos**
- Conta Cosmos DB for NoSQL criada (ex.: `cosmos-anp`)
- Banco: `anp` · Container: `cruzamento_petroleo` (partition key: `/ano`)
- Service Principal com role `Cosmos DB Built-in Data Contributor` no recurso,
  ou identidade gerenciada do workspace Fabric com a mesma role
- Variáveis definidas abaixo (ou via `mssparkutils.credentials`/Key Vault)

In [ ]:
%pip install azure-cosmos==4.9.0 azure-identity==1.19.0 --quiet

In [ ]:
import json

from pyspark.sql.functions import col

COSMOS_ENDPOINT = "https://cosmos-anp.documents.azure.com:443/"
TENANT_ID = "<tenant-id>"
CLIENT_ID = "<client-id-do-service-principal>"
CLIENT_SECRET = mssparkutils.credentials.getSecret(
    "https://kv-anp.vault.azure.net/", "cosmos-sp-secret"
)
DB_NAME = "anp"
CONTAINER_NAME = "cruzamento_petroleo"

## 1. Ler tabelas Delta curadas

In [ ]:
fato_mensal = spark.table("fato_mensal").orderBy("ANO", "MES_NUM")
fato_anual = spark.table("fato_anual").orderBy("ANO")

mensal_por_ano = {}
for r in fato_mensal.collect():
    mensal_por_ano.setdefault(r["ANO"], []).append({
        "mes": int(r["MES_NUM"]),
        "producao_barris": float(r["PRODUCAO_BARRIS"]),
        "importacao_barris": float(r["IMPORTACAO_BARRIS"]),
        "exportacao_barris": float(r["EXPORTACAO_BARRIS"]),
        "balanca_comercial_barris": float(r["BALANCA_COMERCIAL_BARRIS"]),
    })

documentos = []
for r in fato_anual.collect():
    ano = int(r["ANO"])
    def num(campo):
        v = r[campo]
        return None if v is None else float(v)
    documentos.append({
        "id": f"anp-petroleo-{ano}",
        "tipo": "cruzamento_anual_petroleo",
        "fonte": "ANP - Dados Estatisticos / Fase de Desenvolvimento e Producao",
        "ano": ano,
        "producao": {
            "total_barris": num("PRODUCAO_BARRIS"),
            "variacao_yoy_pct": num("PRODUCAO_BARRIS_YOY_PCT"),
        },
        "comercio_exterior": {
            "exportacao_barris": num("EXPORTACAO_BARRIS"),
            "importacao_barris": num("IMPORTACAO_BARRIS"),
            "balanca_comercial_barris": num("BALANCA_COMERCIAL_BARRIS"),
            "exportacao_yoy_pct": num("EXPORTACAO_BARRIS_YOY_PCT"),
            "importacao_yoy_pct": num("IMPORTACAO_BARRIS_YOY_PCT"),
        },
        "fases_contratuais": {
            "campos_em_producao": int(r["CAMPOS_EM_PRODUCAO"]),
            "campos_em_desenvolvimento": int(r["CAMPOS_EM_DESENVOLVIMENTO"]),
            "campos_total_fase_producao": int(r["CAMPOS_TOTAL_FASE"]),
        },
        "indicadores": {
            "cobertura_exportacao_pct": num("COBERTURA_EXPORT_PCT"),
            "dependencia_importacao_pct": num("DEPENDENCIA_IMPORT_PCT"),
            "producao_media_bbl_dia": num("PRODUCAO_BARRIS") / 365.25,
            "producao_por_campo_barris": num("PRODUCAO_BARRIS") / int(r["CAMPOS_EM_PRODUCAO"]),
        },
        "mensal": mensal_por_ano.get(ano, []),
    })

print(f"{len(documentos)} documentos prontos ({', '.join(d['id'] for d in documentos)})")

## 2. Upsert no Cosmos DB for NoSQL (Entra ID)

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import ClientSecretCredential

credencial = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
cliente = CosmosClient(COSMOS_ENDPOINT, credential=credencial)

banco = cliente.create_database_if_not_exists(id=DB_NAME)
container = banco.create_container_if_not_exists(
    id=CONTAINER_NAME,
    partition_key=PartitionKey(path="/ano"),
)

for doc in documentos:
    container.upsert_item(doc)

print(f"Upsert concluído: {len(documentos)} documentos em {DB_NAME}/{CONTAINER_NAME}")

## 3. Smoke test — consulta ao container

In [ ]:
consulta = """
SELECT c.ano,
       c.producao.total_barris AS producao,
       c.comercio_exterior.exportacao_barris AS exportacao,
       c.indicadores.cobertura_exportacao_pct AS cobertura_pct
FROM c ORDER BY c.ano
"""
for item in container.query_items(query=consulta, enable_cross_partition_query=True):
    print(item)